<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Building_a_Simple_Cnn_in_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Core Cnn Layers in pytorch

In [ ]:
import torch
import torch.nn as nn

# Example: A Conv2d layer expecting 3 input channels (e.g., RGB),
# producing 16 output channels using 5x5 filters.
conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)


In [ ]:
# Example: A MaxPool2d layer with a 2x2 window and a stride of 2.
# This will typically halve the height and width of the input.
pool1 = nn.MaxPool2d(kernel_size=2, stride=2)


In [ ]:
# ReLU activation function
relu1 = nn.ReLU()


In [ ]:
# Example: A linear layer taking a flattened vector of 512 features
# and outputting 10 values (e.g., for 10 classes).
fc1 = nn.Linear(in_features=512, out_features=10)


# Defining the Cnn Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F # Often contains activation functions and other utilities

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Layer definitions
        # Convolutional Layer 1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5, stride=1, padding=2)
        # Max Pooling Layer 1
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Convolutional Layer 2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=2)
        # Max Pooling Layer 2
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        # The input features for fc1 depends on the output shape after pooling
        # Input: 28x28 -> Conv1 (padding=2) -> 28x28 -> Pool1 (stride=2) -> 14x14
        # -> Conv2 (padding=2) -> 14x14 -> Pool2 (stride=2) -> 7x7
        # So, the flattened size is 32 channels * 7 height * 7 width = 1568
        self.fc1 = nn.Linear(in_features=32 * 7 * 7, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=10) # Output for 10 classes

    def forward(self, x):
        # Define the data flow through the layers
        # Input x shape: [Batch Size, 1, 28, 28]

        # Apply Conv1, ReLU, Pool1
        x = self.pool1(F.relu(self.conv1(x)))
        # Shape after pool1: [Batch Size, 16, 14, 14]

        # Apply Conv2, ReLU, Pool2
        x = self.pool2(F.relu(self.conv2(x)))
        # Shape after pool2: [Batch Size, 32, 7, 7]

        # Flatten the tensor for the fully connected layers
        # -1 maintains the batch size dimension
        x = x.view(-1, 32 * 7 * 7)
        # Shape after view: [Batch Size, 1568]

        # Apply FC1 and ReLU
        x = F.relu(self.fc1(x))
        # Shape after fc1: [Batch Size, 128]

        # Apply FC2 (output layer, no activation here, typically applied with loss function)
        x = self.fc2(x)
        # Shape after fc2: [Batch Size, 10]
        return x



In [ ]:
# Instantiate the model
model = SimpleCNN()
print(model)

# Create a dummy input tensor (batch of 4 images, 1 channel, 28x28)
# Requires gradient tracking if you intend to train
dummy_input = torch.randn(4, 1, 28, 28)

# Pass the input through the model (forward pass)
output = model(dummy_input)

# Check the output shape
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}") # Expected: [4, 10]


SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

Input shape: torch.Size([4, 1, 28, 28])
Output shape: torch.Size([4, 10])
